In [1]:
"""
CNN from Scratch - Fashion MNIST
UE24CS645BC2 Assignment 1
"""

import numpy as np
import time

# ─────────────────────────────────────────────
# 1. DATA LOADING
# ─────────────────────────────────────────────

CLASS_NAMES = [
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "Sandal","Shirt","Sneaker","Bag","Ankle boot"
]

def load_data():
    from tensorflow.keras.datasets import fashion_mnist
    print("Loading Fashion MNIST via Keras ...")
    (X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
    X_train = X_train.astype(np.float32) / 255.0
    X_test  = X_test.astype(np.float32)  / 255.0
    # Add channel dim → (N, 1, 28, 28)
    X_train = X_train[:, np.newaxis, :, :]
    X_test  = X_test[:, np.newaxis, :, :]
    return X_train, y_train, X_test, y_test


# ─────────────────────────────────────────────
# 2. ACTIVATION FUNCTIONS
# ─────────────────────────────────────────────

def relu(z):
    return np.maximum(0, z)

def relu_backward(dA, z):
    return dA * (z > 0)

def softmax(z):
    # Numerically stable row-wise softmax
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


# ─────────────────────────────────────────────
# 3. CONVOLUTION LAYER
# ─────────────────────────────────────────────

class ConvLayer:
    """
    2-D Convolution Layer (no padding, stride=1).

    Parameters
    ----------
    in_channels  : number of input channels
    out_channels : number of filters (output channels)
    kernel_size  : square kernel size (e.g. 3 → 3×3)
    """
    def __init__(self, in_channels, out_channels, kernel_size, lr=0.01):
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size  = kernel_size
        self.lr           = lr

        # He initialisation
        fan_in = in_channels * kernel_size * kernel_size
        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * np.sqrt(2.0 / fan_in)
        self.b = np.zeros((out_channels, 1))

    # ── Forward ────────────────────────────────
    def forward(self, X):
        """
        X : (batch, C_in, H, W)
        Returns Z : (batch, C_out, H_out, W_out)
        """
        self.X = X                              # cache for backward
        N, C, H, W = X.shape
        K = self.kernel_size
        H_out = H - K + 1
        W_out = W - K + 1

        Z = np.zeros((N, self.out_channels, H_out, W_out), dtype=np.float32)
        for f in range(self.out_channels):
            for i in range(H_out):
                for j in range(W_out):
                    patch = X[:, :, i:i+K, j:j+K]          # (N, C, K, K)
                    Z[:, f, i, j] = np.sum(patch * self.W[f], axis=(1, 2, 3)) + self.b[f]
        self.Z = Z
        return relu(Z)

    # ── Backward ───────────────────────────────
    def backward(self, dA):
        """
        dA : gradient of loss w.r.t. output after ReLU  (N, C_out, H_out, W_out)
        Returns dX : gradient w.r.t. input X
        """
        dZ = relu_backward(dA, self.Z)         # through ReLU

        N, C, H, W = self.X.shape
        K = self.kernel_size
        H_out, W_out = dZ.shape[2], dZ.shape[3]

        dW = np.zeros_like(self.W)
        db = np.zeros_like(self.b)
        dX = np.zeros_like(self.X)

        for f in range(self.out_channels):
            for i in range(H_out):
                for j in range(W_out):
                    patch = self.X[:, :, i:i+K, j:j+K]     # (N, C, K, K)
                    d     = dZ[:, f, i, j]                  # (N,)
                    dW[f] += np.sum(patch * d[:, np.newaxis, np.newaxis, np.newaxis], axis=0)
                    db[f] += np.sum(d)
                    dX[:, :, i:i+K, j:j+K] += self.W[f] * d[:, np.newaxis, np.newaxis, np.newaxis]

        # Gradient descent update
        self.W -= self.lr * dW / N
        self.b -= self.lr * db / N
        return dX


# ─────────────────────────────────────────────
# 4. MAX POOLING LAYER
# ─────────────────────────────────────────────

class MaxPoolLayer:
    """
    2-D Max Pooling (non-overlapping, pool_size × pool_size windows).
    """
    def __init__(self, pool_size=2):
        self.pool_size = pool_size

    def forward(self, X):
        """X : (N, C, H, W)"""
        self.X = X
        N, C, H, W = X.shape
        P = self.pool_size
        H_out = H // P
        W_out = W // P
        out = np.zeros((N, C, H_out, W_out), dtype=np.float32)
        for i in range(H_out):
            for j in range(W_out):
                patch = X[:, :, i*P:(i+1)*P, j*P:(j+1)*P]
                out[:, :, i, j] = np.max(patch, axis=(2, 3))
        return out

    def backward(self, dout):
        """Distribute gradients only to the max-value positions."""
        N, C, H, W = self.X.shape
        P = self.pool_size
        H_out = H // P
        W_out = W // P
        dX = np.zeros_like(self.X)
        for i in range(H_out):
            for j in range(W_out):
                patch = self.X[:, :, i*P:(i+1)*P, j*P:(j+1)*P]   # (N, C, P, P)
                max_vals = np.max(patch, axis=(2, 3), keepdims=True)
                mask = (patch == max_vals)
                # If multiple equal max values, split gradient equally
                count = np.sum(mask, axis=(2, 3), keepdims=True)
                dX[:, :, i*P:(i+1)*P, j*P:(j+1)*P] += (
                    mask / count * dout[:, :, i:i+1, j:j+1]
                )
        return dX


# ─────────────────────────────────────────────
# 5. FLATTEN LAYER
# ─────────────────────────────────────────────

class FlattenLayer:
    def forward(self, X):
        self.shape = X.shape
        return X.reshape(X.shape[0], -1)

    def backward(self, dout):
        return dout.reshape(self.shape)


# ─────────────────────────────────────────────
# 6. FULLY CONNECTED LAYER
# ─────────────────────────────────────────────

class FCLayer:
    """
    Dense (fully-connected) layer with optional ReLU activation.
    The last layer should use activation=None (softmax applied at loss level).
    """
    def __init__(self, in_features, out_features, lr=0.01, activation='relu'):
        self.lr         = lr
        self.activation = activation

        # He init for ReLU; Xavier for softmax output
        scale = np.sqrt(2.0 / in_features) if activation == 'relu' else np.sqrt(1.0 / in_features)
        self.W = np.random.randn(in_features, out_features).astype(np.float32) * scale
        self.b = np.zeros((1, out_features), dtype=np.float32)

    def forward(self, X):
        self.X = X
        self.Z = X @ self.W + self.b
        if self.activation == 'relu':
            return relu(self.Z)
        return self.Z  # raw logits for softmax output layer

    def backward(self, dout):
        if self.activation == 'relu':
            dout = relu_backward(dout, self.Z)
        N = self.X.shape[0]
        dW = self.X.T @ dout / N
        db = np.sum(dout, axis=0, keepdims=True) / N
        dX = dout @ self.W.T
        self.W -= self.lr * dW
        self.b -= self.lr * db
        return dX


# ─────────────────────────────────────────────
# 7. LOSS (Cross-Entropy + Softmax combined)
# ─────────────────────────────────────────────

def cross_entropy_loss(logits, y):
    """
    logits : (N, C) raw scores
    y      : (N,)  integer class labels
    Returns scalar loss and gradient dlogits
    """
    N = logits.shape[0]
    probs = softmax(logits)
    # Clip for numerical stability
    probs_clipped = np.clip(probs, 1e-15, 1.0)
    loss = -np.mean(np.log(probs_clipped[np.arange(N), y]))

    # Gradient of softmax + cross-entropy combined
    dlogits = probs.copy()
    dlogits[np.arange(N), y] -= 1          # ∂L/∂logits
    return loss, dlogits


# ─────────────────────────────────────────────
# 8. CNN MODEL
# ─────────────────────────────────────────────

class CNN:
    """
    Architecture:
      Conv(1→8, 3×3) → ReLU → MaxPool(2) → [13×13×8]
      Conv(8→16, 3×3) → ReLU → MaxPool(2) → [5×5×16 = 400]
      Flatten
      FC(400 → 128) → ReLU
      FC(128 → 10)  → Softmax (via loss)
    """
    def __init__(self, lr=0.01):
        self.conv1   = ConvLayer(1,  8,  3, lr=lr)
        self.pool1   = MaxPoolLayer(2)
        self.conv2   = ConvLayer(8,  16, 3, lr=lr)
        self.pool2   = MaxPoolLayer(2)
        self.flatten = FlattenLayer()
        self.fc1     = FCLayer(400, 128, lr=lr, activation='relu')
        self.fc2     = FCLayer(128,  10, lr=lr, activation=None)

    def forward(self, X):
        out = self.conv1.forward(X)
        out = self.pool1.forward(out)
        out = self.conv2.forward(out)
        out = self.pool2.forward(out)
        out = self.flatten.forward(out)
        out = self.fc1.forward(out)
        out = self.fc2.forward(out)
        return out

    def backward(self, dlogits):
        d = self.fc2.backward(dlogits)
        d = self.fc1.backward(d)
        d = self.flatten.backward(d)
        d = self.pool2.backward(d)
        d = self.conv2.backward(d)
        d = self.pool1.backward(d)
        d = self.conv1.backward(d)

    def predict(self, X):
        logits = self.forward(X)
        return np.argmax(softmax(logits), axis=1)


# ─────────────────────────────────────────────
# 9. TRAINING FUNCTION
# ─────────────────────────────────────────────

def train(model, X_train, y_train, X_val, y_val,
          epochs=10, batch_size=64):

    n = X_train.shape[0]
    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        # Shuffle training data
        idx = np.random.permutation(n)
        X_s, y_s = X_train[idx], y_train[idx]

        epoch_loss = 0.0
        epoch_correct = 0
        t0 = time.time()

        for start in range(0, n, batch_size):
            Xb = X_s[start:start+batch_size]
            yb = y_s[start:start+batch_size]

            logits = model.forward(Xb)
            loss, dlogits = cross_entropy_loss(logits, yb)
            model.backward(dlogits)

            epoch_loss    += loss * len(yb)
            preds          = np.argmax(logits, axis=1)
            epoch_correct += np.sum(preds == yb)

        train_loss = epoch_loss / n
        train_acc  = epoch_correct / n

        # Validation accuracy (in chunks to save memory)
        val_preds = []
        for start in range(0, X_val.shape[0], batch_size):
            val_preds.append(model.predict(X_val[start:start+batch_size]))
        val_acc = np.mean(np.concatenate(val_preds) == y_val)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        elapsed = time.time() - t0
        print(f"Epoch {epoch:2d}/{epochs}  "
              f"loss={train_loss:.4f}  "
              f"train_acc={train_acc*100:.2f}%  "
              f"val_acc={val_acc*100:.2f}%  "
              f"({elapsed:.1f}s)")

    return history


# ─────────────────────────────────────────────
# 10. EVALUATION FUNCTION
# ─────────────────────────────────────────────

def evaluate(model, X_test, y_test, batch_size=64):
    preds = []
    for start in range(0, X_test.shape[0], batch_size):
        preds.append(model.predict(X_test[start:start+batch_size]))
    preds = np.concatenate(preds)
    acc   = np.mean(preds == y_test)

    # Per-class accuracy
    print(f"\n{'='*50}")
    print(f"Test Accuracy: {acc*100:.2f}%")
    print(f"{'='*50}")
    print(f"\n{'Class':<20} {'Correct':>8} {'Total':>8} {'Acc':>8}")
    print("-" * 50)
    for c, name in enumerate(CLASS_NAMES):
        mask  = y_test == c
        total = mask.sum()
        corr  = (preds[mask] == c).sum()
        print(f"{name:<20} {corr:>8} {total:>8} {corr/total*100:>7.1f}%")
    print("="*50)
    return acc


# ─────────────────────────────────────────────
# 11. MAIN
# ─────────────────────────────────────────────

if __name__ == "__main__":
    np.random.seed(42)

    print("Loading Fashion MNIST ...")
    X_train, y_train, X_test, y_test = load_data()

    # Use a small validation split from training data
    val_size = 5000
    X_val, y_val       = X_train[:val_size], y_train[:val_size]
    X_train, y_train   = X_train[val_size:], y_train[val_size:]

    # Use a subset of training data to keep it practical on CPU
    # Remove the next two lines to train on full 55k images (much slower)
    TRAIN_SUBSET = 10000
    X_train, y_train = X_train[:TRAIN_SUBSET], y_train[:TRAIN_SUBSET]

    print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

    print("\nBuilding CNN ...")
    model = CNN(lr=0.01)

    print("\nTraining ...\n")
    history = train(model, X_train, y_train, X_val, y_val,
                    epochs=10, batch_size=64)

    print("\nEvaluating on test set ...")
    evaluate(model, X_test, y_test)

Loading Fashion MNIST ...
Loading Fashion MNIST via Keras ...
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train: (10000, 1, 28, 28)  Val: (5000, 1, 28, 28)  Test: (10000, 1, 28, 28)

Building CNN ...

Training ...

Epoch  1/10  loss=1.3650  train_acc=53.15%  val_acc=61.26%  (150.9s)
Epoch  2/10  loss=0.8872  train_acc=66.22%  val_acc=69.32%  (150.2s)
Epoch  3/10  loss=0.7961  train_acc=70.16%  val_acc=65.32%  (146.9s)
Epoch  4/10  loss=0.7288  train_acc=72.76%  val_acc=71.92%  (147.8s)
Epoch  5/10  loss=0.6903  train_acc=74.32%  val_acc=65.88%  (147.2s)
Epoch  6/10  loss=0.6544  train_acc=75.31%  val_acc=74.66%  (146.9s)
Epoch  7/10  loss=0.6230  train_acc=76.62%  val_acc=76.28%  (147.4s)
Epoch  8/10  loss=0.5995  train_acc=77.58%  val_acc=71.68%  (147.7s)
Epoch  9/10  loss=0.5821  train_acc=78.08%  val_acc=75.36%  (146.6s)
Epoch 10/10  loss=0.